# Klassik RLHF: reward model + PPO, SFT v3 LoRA (Qwen2.5-1.5B) ustida

DPO v2 bilan bir xil chosen/rejected juftliklari (`dpo/data_v2`), lekin klassik yo'l bilan:

1. **Reward model** (Qwen2.5-0.5B + LoRA + skalyar head) juftliklardan o'rganadi: javobga bitta ball beradi.
2. **PPO** (online): SFT v3 policy har bir promptga javob yozadi, reward model baholaydi, policy yangilanadi. Xotirada 4 ta model: policy, reference, reward, value.
3. 68 ta test savolida baholanadi va SFT v3 (41/68), DPO v2 (44/68) bilan solishtiriladi.

**Boshlashdan oldin:**

- **Settings -> Accelerator -> GPU T4 x2** (ikkita GPU kerak: policy GPU 0'da, qolgan uchta model GPU 1'da), **Internet -> On**
- **SFT v3 adapter:** **Add Input** orqali DPO'da ishlatilgan `sft-v3-adapters` datasetini qo'shing (yoki `sft_v3/sft_v3_adapters.zip` ni yangi dataset qilib yuklang).
- **Save Version -> Save & Run All (Commit)** bilan ishga tushiring: natija **Output** bo'limida saqlanadi.

## 1. GPU borligini tekshirish

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Loyiha va base modellarni yuklab olish

Policy uchun Qwen2.5-1.5B, reward va value model uchun Qwen2.5-0.5B (tokenizer bir xil).

In [ ]:
%cd /kaggle/working
!rm -rf Post_Training
!GIT_LFS_SKIP_SMUDGE=1 git clone -q https://github.com/davronbek-malikov/Post_Training.git
%cd Post_Training
!pip -q install -U transformers peft bitsandbytes accelerate
!pip -q uninstall -y torchao  # Kaggle'dagi eski torchao peft bilan adapter yuklashni buzadi

from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen2.5-1.5B", local_dir="Qwen2.5-1.5B")
snapshot_download("Qwen/Qwen2.5-0.5B", local_dir="Qwen2.5-0.5B")
!ls -lh Qwen2.5-1.5B/*.safetensors Qwen2.5-0.5B/*.safetensors
!wc -l dpo/data_v2/train.jsonl dpo/data_v2/val.jsonl dpo/data/prompts.jsonl

## 3. SFT v3 LoRA adapter

Avval yuklangan dataset ichidan qidiriladi; topilmasa, qaytadan o'qitiladi.

In [ ]:
import glob, os, shutil, subprocess, zipfile
target = "chat_ui/adapters/sft-lora-v3"
found = glob.glob("/kaggle/input/**/sft-lora-v3/adapter_config.json", recursive=True)
if not found:   # zip ochilmagan holda yuklangan bo'lsa
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        with zipfile.ZipFile(z) as f:
            names = [n for n in f.namelist() if "sft-lora-v3/" in n]
            if names:
                f.extractall("/kaggle/working/unzipped", members=names)
    found = glob.glob("/kaggle/working/unzipped/**/sft-lora-v3/adapter_config.json", recursive=True)
if found:
    shutil.copytree(os.path.dirname(found[0]), target, dirs_exist_ok=True)
    print("SFT v3 adapter topildi:", os.path.dirname(found[0]))
else:
    print("Adapter topilmadi: SFT v3 LoRA qaytadan o'qitiladi...")
    subprocess.run(["python", "sft_v2/train.py", "--method", "lora", "--data", "sft_v3/data",
                    "--name", "sft-lora-v3", "--epochs", "3", "--rank", "64"], check=True)
print(os.listdir(target))

## 4. Reward model

DPO v2 juftliklarida o'qitiladi. Loss: `-log σ(r(chosen) − r(rejected))`. Tekshiring:
- `val acc` 0.5 dan ancha yuqori bo'lishi kerak (0.5 — tasodifiy). Val atigi 18 ta juftlik, shuning uchun shovqinli;
- **qisqa javobni afzal ko'rdi** foizi data'dagi "chosen qisqaroq" foiziga yaqin bo'lishi kerak. Ancha ko'p bo'lsa, reward model uzunlikni o'rgangan va PPO ham qisqa javoblarga o'tib ketadi.

In [ ]:
!python rlhf/train_reward.py

## 5. PPO

Har qadamda 8 ta prompt. Kuzating:
- 1-qadamda `KL` ~0 (policy hali reference bilan bir xil);
- `RM ball` o'sishi kerak — PPO aynan shuni oshiradi;
- `qoida bali` (`dpo/label.py`, PPO uni ko'rmaydi) ham o'sishi yoki joyida qolishi kerak. RM ball o'sib, qoida bali tushsa — **reward hacking**;
- `KL` asta o'sadi; juda katta bo'lib ketsa (masalan 10+), policy SFT modeldan juda uzoqlashgan;
- `uzunlik` va `to'xtadi`: javoblar keskin qisqarsa yoki to'xtamay qolsa — muammo.

In [ ]:
!python rlhf/train_ppo.py

In [ ]:
!rm -f /kaggle/working/rlhf_results.zip
!zip -q -r /kaggle/working/rlhf_results.zip chat_ui/adapters/sft-lora-rlhf rlhf/logs rlhf/models/1_base-0.5b/meta.json rlhf/models/1_base-0.5b/training_log.json
!ls -lh /kaggle/working/rlhf_results.zip

## 6. Baholash: 68 ta test savoli

SFT v3 LoRA: 41/68. DPO v1: 40/68. DPO v2: 44/68.

In [ ]:
!python eval/run_eval.py --base Qwen2.5-1.5B --model sft-lora-rlhf | tail -12
!zip -q /kaggle/working/rlhf_results.zip eval/results/sft-lora-rlhf.json

## 7. Yuklab olish

Notebook tugagach: notebook sahifasi -> **Output** -> `rlhf_results.zip` -> **Download**. Uni loyihaning `rlhf/` papkasiga qo'ying.